# 19 · LSTM con y sin actividad solar
Greta Solange Díaz Vega · Seminario de Tesis II · UNI

Redes de ocho unidades con ventanas de 12 meses y horizontes de 3, 6, 12 y 24 meses. La validación 1995–2009 elige la duración de entrenamiento. El ajuste final utiliza objetivos anteriores a 2010. Una sola semilla por experimento. La ejecución completa requiere varios minutos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]

horizontes = [3,6,12,24]
numero = 19
print(df.index.min().date(), df.index.max().date(), len(df), "meses")

1950-01-01 2026-05-01 917 meses


## Análisis

In [2]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
filas = []
for indice in ["ICEN","ONI"]:
    for solar in [False,True]:
        variables = [indice,"SSN"] if solar else [indice]
        valores = df[variables].to_numpy()
        indices = np.arange(12,len(df))
        # Ventanas desde t-12 hasta t-1. Una red por horizonte.
        ventanas = np.stack([valores[t-12:t] for t in indices])
        for h in horizontes:
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(2026)
            validos = indices+h < len(df)
            t = indices[validos]
            X = ventanas[validos]
            y = df[indice].to_numpy()[t+h]
            destino = df.index[t+h]
            train = destino < "1995-01-01"
            val = (df.index[t] >= "1995-01-01") & (destino < "2010-01-01")
            test = df.index[t] >= "2010-01-01"
            escalador = StandardScaler().fit(X[train].reshape(-1,len(variables)))
            X = escalador.transform(X.reshape(-1,len(variables))).reshape(X.shape).astype("float32")
            def red():
                modelo = tf.keras.Sequential([tf.keras.Input((12,len(variables))),
                    tf.keras.layers.LSTM(8),tf.keras.layers.Dense(1)])
                modelo.compile(optimizer="adam",loss="mse")
                return modelo
            def epoca(modelo, mascara):
                posiciones = np.flatnonzero(mascara)
                for i in range(0,len(posiciones),32):
                    lote = posiciones[i:i+32]
                    modelo.train_on_batch(X[lote],y[lote])
            modelo = red()
            errores = []
            for i in range(15):
                epoca(modelo,train)
                pred = modelo(X[val],training=False).numpy().ravel()
                errores.append(np.mean((y[val]-pred)**2))
            mejor_epoca = int(np.argmin(errores))+1
            tf.keras.utils.set_random_seed(2026)
            modelo = red()
            for i in range(mejor_epoca):
                epoca(modelo,destino < "2010-01-01")
            pred = modelo(X[test],training=False).numpy().ravel()
            filas.extend(zip([indice]*len(pred),[h]*len(pred),[solar]*len(pred),df.index[t[test]],y[test],pred))


## Evaluación en los mismos meses
MAE es el error absoluto medio. RMSE da más peso a los errores grandes. Compara las filas con y sin SSN del mismo índice y horizonte.

In [3]:
pred = pd.DataFrame(filas,columns=["indice","h","con_SSN","origen","observado","pronostico"])
pred["destino"] = [fecha+pd.DateOffset(months=int(h)) for fecha,h in zip(pred.origen,pred.h)]
assert not pred.duplicated(["indice","h","con_SSN","origen"]).any()
assert pred.groupby(["indice","h","origen"]).size().eq(2).all()
assert np.isfinite(pred.pronostico).all()
pred["error"] = pred.pronostico-pred.observado
metricas = pred.groupby(["indice","h","con_SSN"]).error.agg(
    N="size", MAE=lambda e:e.abs().mean(), RMSE=lambda e:np.sqrt((e**2).mean()))
pred.to_csv(f"{numero}_pronosticos.csv",index=False)
metricas.to_csv(f"{numero}_metricas.csv")
print(metricas.round(3).to_string())


                     N    MAE   RMSE
indice h  con_SSN                   
ICEN   3  False    194  0.723  0.916
          True     194  0.717  0.921
       6  False    191  0.818  1.022
          True     191  0.803  1.018
       12 False    185  0.847  1.051
          True     185  0.798  1.013
       24 False    173  0.868  1.047
          True     173  0.829  1.029
ONI    3  False    194  0.555  0.725
          True     194  0.590  0.781
       6  False    191  0.618  0.812
          True     191  0.668  0.866
       12 False    185  0.605  0.779
          True     185  0.638  0.820
       24 False    173  0.578  0.756
          True     173  0.578  0.773
